# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arslaniqbalwah/flyrank-ml-internship-arslan/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Based on our signal audit, newer pages actually decay faster than older ones. My baseline rule prioritizes pages with high traffic (impressions) but gives a 1.5x score multiplier to pages younger than 365 days.

Rule: Score = impressions_90d (x1.5 if age < 365 days).

Reason Code: high_impact_new_decay (for young, high-traffic pages) and standard_review (for the rest).

Action: review_for_refresh.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


print("Rule: Prioritize high-traffic pages, with a bonus weight for pages under 1 year old.")
print("Reason codes: 'high_impact_new_decay' and 'standard_review'.")

Rule: Prioritize high-traffic pages, with a bonus weight for pages under 1 year old.
Reason codes: 'high_impact_new_decay' and 'standard_review'.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Here I am calculating the baseline score, assigning the reason codes, and sorting the dataframe to create our ranked review queue. The results are saved to the required CSV file.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


import pandas as pd
import os

url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# Calculate baseline score
df['baseline_score'] = df['impressions_90d'].fillna(0)
df.loc[df['content_age_days'] < 365, 'baseline_score'] *= 1.5

# Assign reason codes and action
df['reason_code'] = 'standard_review'
df.loc[(df['content_age_days'] < 365) & (df['impressions_90d'] > 1000), 'reason_code'] = 'high_impact_new_decay'
df['action'] = 'review_for_refresh'

# Sort to create the ranked queue
queue = df.sort_values('baseline_score', ascending=False).copy()

# Create folder and save CSV
os.makedirs('work/outputs', exist_ok=True)
queue.to_csv('work/outputs/baseline_action_score.csv', index=False)

print(f"Success: Saved {len(queue)} rows to work/outputs/baseline_action_score.csv")
display(queue[['content_id', 'baseline_score', 'reason_code', 'action']].head())

/tmp/ipykernel_1064/4118859817.py:13: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[ 5704.5 18871.5 28710.  ...  1141.5  9504.   6351. ]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[df['content_age_days'] < 365, 'baseline_score'] *= 1.5


Success: Saved 30000 rows to work/outputs/baseline_action_score.csv


,content_id,baseline_score,reason_code,action
19636,content_2cb567c3c89b,746590.5,high_impact_new_decay,review_for_refresh
29400,content_2dba2b1f9536,665151.0,high_impact_new_decay,review_for_refresh
13537,content_2c2606c5d176,521098.5,high_impact_new_decay,review_for_refresh
6653,content_5fe46e04994d,517715.0,standard_review,review_for_refresh
17812,content_aaef01a50def,517109.0,standard_review,review_for_refresh


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Looking at the top 20 pages, they are heavily dominated by the high_impact_new_decay reason code.

Action: Review for content refresh.

Why it's there: These pages drive massive traffic and are relatively new, making them high-priority targets.

What would make it wrong: This simple rule might just be catching viral articles or seasonal holiday pages that had a massive temporary spike and are now naturally settling down. A fixed rule can't tell the difference.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


print("Top 20 Ranked Pages for Review:")
top_20 = queue[['content_id', 'baseline_score', 'reason_code', 'content_age_days', 'impressions_90d']].head(20)
display(top_20)

Top 20 Ranked Pages for Review:


,content_id,baseline_score,reason_code,content_age_days,impressions_90d
19636,content_2cb567c3c89b,746590.5,high_impact_new_decay,153,497727
29400,content_2dba2b1f9536,665151.0,high_impact_new_decay,299,443434
13537,content_2c2606c5d176,521098.5,high_impact_new_decay,362,347399
6653,content_5fe46e04994d,517715.0,standard_review,537,517715
17812,content_aaef01a50def,517109.0,standard_review,445,517109
26844,content_8c19996aa890,509252.0,standard_review,445,509252
14090,content_44e481c8f55b,469041.0,high_impact_new_decay,112,312694
26531,content_cb112fce36be,464865.0,high_impact_new_decay,126,309910
21819,content_4c36c775b818,463103.0,standard_review,445,463103
3394,content_36ff89c8214e,442645.5,high_impact_new_decay,144,295097


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

The biggest weakness here is that our baseline score is just a blunt instrument. It blindly multiplies impressions by 1.5 for young pages, which means a perfectly healthy new page with high traffic gets flagged as "urgent" even if it isn't decaying at all.

Leakage Check: I confirmed that the actual proxy label (trend_direction) was NOT used in calculating the score. The score only uses historical impressions and age. No future windows leaked in.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Verify that the target label was not used in our baseline logic
leak_check = 'trend_direction' in ['impressions_90d', 'content_age_days']
print(f"Did we accidentally use the target label in our score logic? {leak_check}")

print("Weak picks logic: A healthy viral page will score dangerously high here just because it's new and popular.")


Did we accidentally use the target label in our score logic? False
Weak picks logic: A healthy viral page will score dangerously high here just because it's new and popular.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.